# Imports y configuración

In [53]:
import random
import time
import cv2
import numpy as np
import mediapipe as mp
from PIL import Image, ImageDraw, ImageFont
import os
import tempfile
import wave

try:
    import winsound
except ImportError:      # solo existe en Windows
    winsound = None

# Cámara
CAMERA_INDEX = 0
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# Rutas de modelos y fuente
FACE_MODEL_PATH = "face_landmarker.task"
HAND_MODEL_PATH = "hand_landmarker.task"
FONT_PATH = "C:/Windows/Fonts/arial.ttf"

# Índices de landmarks de la malla facial (izquierda/derecha según se ven en la imagen)
LM_NOSE_TIP = 1
LM_FOREHEAD = 10
LM_CHIN = 152
LM_FACE_LEFT = 234
LM_FACE_RIGHT = 454
LM_EYE_LEFT_OUTER = 33
LM_EYE_RIGHT_OUTER = 263

# Colores de cada jugador en RGB (J1 a la izquierda, J2 a la derecha)
PLAYER_COLORS = [(255, 170, 0), (0, 200, 255)]

# Más índices de la malla facial: base de la nariz y labio superior
LM_NOSE_BOTTOM = 2
LM_UPPER_LIP = 0

# Ajustes de los accesorios (proporciones respecto al ancho de la cara)
HAT_WIDTH_FACTOR = 1.45        # ancho del ala del sombrero
HAT_LIFT = 0.22                 # sube (+) o baja (-) el sombrero
MUSTACHE_WIDTH_FACTOR = 0.62   # ancho del bigote

# Colores BGR de cada sombrero: (cuerpo, cinta)
HAT_COLORS = [((45, 95, 150), (30, 30, 40)), ((55, 55, 60), (40, 40, 170))]

# Gestos de cara (el valor de cada uno va de 0 a 1)
KISS_THRESHOLD = 0.80        # morritos: reposo hasta 0.55, beso 0.99-1.00
BROWS_UP_THRESHOLD = 0.50    # cejas: reposo 0.10-0.20, levantadas 0.80-0.90

# Frames seguidos que debe mantenerse un gesto para darlo por válido
GESTURE_HOLD_FRAMES = 2

# Manos
USE_HANDS = True               # False desactiva el detector de manos si va lento

# Pistola de frente (curl: recorrido del dedo / distancia directa, recto ~1)
FINGER_STRAIGHT_MAX = 1.30   # dedo recto si su curl queda por debajo
FINGER_CURLED_MIN = 1.70     # dedo doblado si su curl queda por encima

# Pistola de lado (ratio: distancia punta-muñeca / articulación media-muñeca)
FINGER_EXTENDED_RATIO = 1.05 # el índice debe superar este ratio
POINTING_MARGIN = 0.15       # y superar al mayor de los otros tres dedos por esta diferencia

# Juego
ROUNDS_TO_WIN = 2                # mejor de 3
ANNOUNCE_SECONDS = 2.5           # tiempo que se avisa del gesto (modo 4)
WAIT_RANGE = (2.5, 6.0)          # espera aleatoria antes de "¡FUEGO!", en segundos
DRAW_TIMEOUT = 5.0               # si nadie dispara en este tiempo se repite la ronda
RESULT_SECONDS = 3.0             # tiempo que se muestra el resultado de cada ronda
MIN_REACTION_SECONDS = 0.10      # reaccionar en menos de esto se considera anticipación

# Efectos y ambiente
SEPIA_STRENGTH = 0.55              # 0 = color original, 1 = sepia completo
VIGNETTE_STRENGTH = 0.55           # oscurecimiento de las esquinas
DUST_COUNT = 40                    # partículas de polvo
TUMBLEWEED_DELAY = (8.0, 15.0)     # segundos entre bolas de paja
FLASH_SECONDS = 0.25               # duración del destello al disparar
BANG_SECONDS = 0.9                 # tiempo que se muestra la explosión
HAT_GRAVITY = 900                  # píxeles/s² del sombrero al salir volando

# Sonido
SOUND_ENABLED = True               # False desactiva todos los sonidos
SOUND_VOLUME = 0.6                 # volumen de 0 a 1
SOUND_RATE = 22050                 # muestras por segundo de los sonidos generados
MUSIC_VOLUME = 0.3                 # la música va más baja que los efectos
MUSIC_BPM = 108                    # velocidad de la música de fondo
FANFARE_SECONDS = 1.6               # la fanfarria dura 1.5 s; la música empieza justo después


# Texto con Pillow

In [54]:
# Cache de fuentes por tamaño para no cargar el archivo en cada frame
_font_cache = {}

def get_font(size):
    if size not in _font_cache:
        _font_cache[size] = ImageFont.truetype(FONT_PATH, size)
    return _font_cache[size]

# Los textos se acumulan en esta lista y se pintan todos juntos al final del frame
_text_queue = []

# Pide un texto centrado en horizontal. Si center_x es None se centra en el frame.
# top_margin es la distancia desde el borde superior hasta lo alto del texto. Color en RGB.
def queue_text(text, size, color, top_margin, center_x=None, stroke_width=2):
    _text_queue.append((text, size, color, top_margin, center_x, stroke_width))

# Dibuja todos los textos pendientes con una sola conversión OpenCV <-> Pillow
def flush_text(frame):
    if not _text_queue:
        return frame

    w = frame.shape[1]
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)

    for text, size, color, top_margin, center_x, stroke_width in _text_queue:
        font = get_font(size)
        # Caja que ocupa el texto, incluido el contorno
        left, top, right, bottom = draw.textbbox((0, 0), text, font=font, stroke_width=stroke_width)
        text_width = right - left

        cx = w // 2 if center_x is None else center_x
        x = cx - text_width // 2 - left
        y = top_margin - top

        draw.text((x, y), text, font=font, fill=color,
                  stroke_width=stroke_width, stroke_fill=(0, 0, 0))

    _text_queue.clear()
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

# Detector de caras

In [55]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

face_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=FACE_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=2,
    output_face_blendshapes=True,
)

# Devuelve las caras ordenadas de izquierda a derecha: la primera es J1 y la segunda J2.
# El timestamp debe crecer en cada llamada (lo exige el modo VIDEO).
def detect_players(landmarker, frame, timestamp_ms):
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    faces = []
    for i, face_landmarks in enumerate(result.face_landmarks):
        # Landmarks normalizados (0-1) pasados a píxeles
        points = np.array([(lm.x * w, lm.y * h) for lm in face_landmarks], dtype=np.float32)
        # La profundidad usa la misma escala que x, así que se multiplica también por w
        depth = np.array([lm.z * w for lm in face_landmarks], dtype=np.float32)
        blendshapes = {c.category_name: c.score for c in result.face_blendshapes[i]}
        faces.append({"points": points, "depth": depth, "blendshapes": blendshapes})

    # Se asigna el jugador por la posición horizontal de la nariz
    faces.sort(key=lambda f: f["points"][LM_NOSE_TIP][0])
    return faces

# Detector de manos

In [ ]:
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# Hasta 4 manos: cada jugador podría enseñar las dos
hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=4,
)

# Devuelve las manos detectadas, con sus puntos en píxeles (para dibujar)
# y en coordenadas 3D reales (para medir los dedos sin depender de la orientación)
def detect_hands(landmarker, frame, timestamp_ms):
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    hands = []
    for i, hand_landmarks in enumerate(result.hand_landmarks):
        points = np.array([(lm.x * w, lm.y * h) for lm in hand_landmarks], dtype=np.float32)
        world = np.array([(lm.x, lm.y, lm.z) for lm in result.hand_world_landmarks[i]],
                         dtype=np.float32)
        hands.append({"points": points, "world": world})
    return hands

# Reparte las manos entre los jugadores: cada mano va al jugador cuya cara
# está más cerca en horizontal (se mide con la muñeca)
def assign_hands(hands, faces):
    assigned = [[] for _ in faces]

    # Sin caras no hay jugador al que asignar las manos
    if not faces:
        return assigned

    for hand in hands:
        wrist_x = hand["points"][0][0]
        distances = [abs(wrist_x - face["points"][LM_NOSE_TIP][0]) for face in faces]
        assigned[int(np.argmin(distances))].append(hand)
    return assigned

# Junta la cara y las manos de cada jugador en un solo diccionario
def build_players(faces, hands):
    hands_per_player = assign_hands(hands, faces)
    return [{"face": face, "hands": hs} for face, hs in zip(faces, hands_per_player)]

def draw_hand_landmarks(frame, hand, color_rgb):
    for x, y in hand["points"]:
        cv2.circle(frame, (int(x), int(y)), 3, color_rgb[::-1], -1)

# Accesorios

In [ ]:
# Medidas de los sprites (en píxeles del propio dibujo)
HAT_BRIM_WIDTH = 250
HAT_ANCHOR = (130, 150)        # centro del ala, es lo que se clava en la frente
MUSTACHE_WIDTH = 184
MUSTACHE_ANCHOR = (100, 36)    # centro del bigote
OUTLINE_COLOR = (20, 20, 25)

# Redondea un polígono cerrado (algoritmo de Chaikin) para que no quede anguloso
def smooth_polygon(points, iterations=2):
    pts = np.array(points, dtype=np.float32)
    for _ in range(iterations):
        nxt = np.roll(pts, -1, axis=0)
        smoothed = np.empty((len(pts) * 2, 2), dtype=np.float32)
        smoothed[0::2] = 0.75 * pts + 0.25 * nxt
        smoothed[1::2] = 0.25 * pts + 0.75 * nxt
        pts = smoothed
    return pts

# Rellena una forma y le dibuja el contorno oscuro (estilo cartoon)
def draw_shape(sprite, points, fill_color, thickness=3):
    pts = np.round(points).astype(np.int32)
    cv2.fillPoly(sprite, [pts], fill_color + (255,), cv2.LINE_AA)
    cv2.polylines(sprite, [pts], True, OUTLINE_COLOR + (255,), thickness, cv2.LINE_AA)

# Sombrero: ala, copa con hendidura y cinta, sobre fondo transparente (BGRA)
def make_hat_sprite(body_color, band_color):
    sprite = np.zeros((190, 260, 4), dtype=np.uint8)

    # Ala: elipse con las puntas curvadas hacia arriba
    brim = cv2.ellipse2Poly((130, 150), (125, 22), 0, 0, 360, 5).astype(np.float32)
    brim[:, 1] -= 20 * (np.abs(brim[:, 0] - 130) / 125) ** 3
    draw_shape(sprite, brim, body_color)

    # Copa: parte de arriba con la hendidura del centro y base curva
    top = np.array([(82, 150), (79, 105), (86, 68), (103, 48), (120, 46), (130, 62),
                    (140, 46), (157, 48), (174, 68), (181, 105), (178, 150)])
    base = cv2.ellipse2Poly((130, 150), (48, 10), 0, 0, 180, 10)
    draw_shape(sprite, smooth_polygon(np.vstack([top, base])), body_color)

    # Cinta: franja entre dos arcos paralelos
    arc_low = cv2.ellipse2Poly((130, 150), (48, 10), 0, 0, 180, 10)
    arc_high = cv2.ellipse2Poly((130, 132), (48, 10), 0, 0, 180, 10)
    draw_shape(sprite, np.vstack([arc_low, arc_high[::-1]]).astype(np.float32), band_color)

    return sprite

# Bigote de manillar: se define la mitad izquierda y se refleja para la derecha
def make_mustache_sprite():
    sprite = np.zeros((64, 200, 4), dtype=np.uint8)
    half = np.array([(100, 30), (80, 24), (60, 28), (42, 38), (28, 40), (18, 34),
                     (10, 24), (8, 12), (16, 20), (22, 32), (30, 46), (52, 52),
                     (74, 50), (92, 46), (100, 42)], dtype=np.float32)
    mirror = half[::-1].copy()
    mirror[:, 0] = 200 - mirror[:, 0]
    draw_shape(sprite, smooth_polygon(np.vstack([half, mirror])), (35, 45, 60))
    return sprite

# Los sprites se dibujan una sola vez, no en cada frame
HAT_SPRITES = [make_hat_sprite(body, band) for body, band in HAT_COLORS]
MUSTACHE_SPRITE = make_mustache_sprite()

# Pega un sprite con transparencia: rota y escala alrededor del ancla,
# y después lleva el ancla al punto destino (target) del frame.
# Solo se procesa la zona que ocupa el sprite, no el frame entero
def blend_sprite(frame, sprite, anchor, target, scale, angle):
    h, w = frame.shape[:2]
    # En OpenCV un ángulo positivo gira en sentido antihorario, de ahí el signo menos
    M = cv2.getRotationMatrix2D(anchor, -angle, scale)
    M[0, 2] += target[0] - anchor[0]
    M[1, 2] += target[1] - anchor[1]

    # Caja que ocupa el sprite ya transformado: se mueven sus 4 esquinas
    sprite_h, sprite_w = sprite.shape[:2]
    corners = np.array([[0, 0, 1], [sprite_w, 0, 1], [0, sprite_h, 1], [sprite_w, sprite_h, 1]],
                       dtype=np.float32)
    moved = corners @ M.T
    x0 = max(int(np.floor(moved[:, 0].min())) - 1, 0)
    y0 = max(int(np.floor(moved[:, 1].min())) - 1, 0)
    x1 = min(int(np.ceil(moved[:, 0].max())) + 1, w)
    y1 = min(int(np.ceil(moved[:, 1].max())) + 1, h)
    if x1 <= x0 or y1 <= y0:
        return  # el sprite queda fuera de la imagen

    # Se traslada la transformación para dibujar solo dentro de esa caja
    M[0, 2] -= x0
    M[1, 2] -= y0
    warped = cv2.warpAffine(sprite, M, (x1 - x0, y1 - y0), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    region = frame[y0:y1, x0:x1]
    alpha = warped[:, :, 3:4].astype(np.float32) / 255.0
    region[:] = (warped[:, :, :3] * alpha + region * (1 - alpha)).astype(np.uint8)

# Ancho de la cara medido en 3D (para que no cambie al girar la cabeza)
# e inclinación de la cabeza en grados según la línea de los ojos
def get_face_metrics(face):
    pts, depth = face["points"], face["depth"]
    left = np.array([*pts[LM_FACE_LEFT], depth[LM_FACE_LEFT]])
    right = np.array([*pts[LM_FACE_RIGHT], depth[LM_FACE_RIGHT]])
    width = np.linalg.norm(right - left)

    eye_l, eye_r = pts[LM_EYE_LEFT_OUTER], pts[LM_EYE_RIGHT_OUTER]
    roll = np.degrees(np.arctan2(eye_r[1] - eye_l[1], eye_r[0] - eye_l[0]))
    return width, roll

# Posición, escala e inclinación del sombrero sobre una cara
def get_hat_transform(face):
    width, roll = get_face_metrics(face)

    # Vector que apunta hacia la coronilla, girado según la inclinación
    rad = np.radians(roll)
    up = np.array([np.sin(rad), -np.cos(rad)])

    pos = face["points"][LM_FOREHEAD] + up * width * HAT_LIFT
    scale = width * HAT_WIDTH_FACTOR / HAT_BRIM_WIDTH
    return pos, scale, roll

def draw_accessories(frame, face, hat_sprite, with_hat=True):
    pts = face["points"]
    width, roll = get_face_metrics(face)

    # Bigote entre la base de la nariz y el labio superior
    mustache_pos = (pts[LM_NOSE_BOTTOM] + pts[LM_UPPER_LIP]) / 2
    mustache_scale = width * MUSTACHE_WIDTH_FACTOR / MUSTACHE_WIDTH
    blend_sprite(frame, MUSTACHE_SPRITE, MUSTACHE_ANCHOR, mustache_pos, mustache_scale, roll)

    # Sombrero con el centro del ala sobre la frente (no se dibuja si ha salido volando)
    if with_hat:
        hat_pos, hat_scale, hat_roll = get_hat_transform(face)
        blend_sprite(frame, hat_sprite, HAT_ANCHOR, hat_pos, hat_scale, hat_roll)

# Puntos de referencia que se usan para colocar los accesorios.
# Con all_points=True se dibuja la malla completa de 478 puntos
def draw_landmarks(frame, face, color_rgb, all_points=False):
    pts = face["points"]
    bgr = color_rgb[::-1]

    if all_points:
        for x, y in pts:
            cv2.circle(frame, (int(x), int(y)), 1, bgr, -1)
        return

    for lm_id in (LM_NOSE_TIP, LM_FOREHEAD, LM_CHIN, LM_FACE_LEFT, LM_FACE_RIGHT,
                  LM_EYE_LEFT_OUTER, LM_EYE_RIGHT_OUTER, LM_NOSE_BOTTOM, LM_UPPER_LIP):
        x, y = pts[lm_id]
        cv2.circle(frame, (int(x), int(y)), 4, bgr, -1)

# Gestos

In [58]:
# (punta, articulación media) de cada dedo en la malla de 21 puntos de la mano
FINGER_JOINTS = {"index": (8, 6), "middle": (12, 10), "ring": (16, 14), "pinky": (20, 18)}

# (base, articulación media, articulación de la punta, punta) de cada dedo
FINGER_CHAINS = {"index": (5, 6, 7, 8), "middle": (9, 10, 11, 12),
                 "ring": (13, 14, 15, 16), "pinky": (17, 18, 19, 20)}

# Cuánto está estirado cada dedo: distancia punta-muñeca entre distancia
# articulación media-muñeca. Estirado sale claramente por encima de 1 y doblado por debajo
def finger_ratios(hand):
    world = hand["world"]
    wrist = world[0]
    return {name: np.linalg.norm(world[tip] - wrist) / np.linalg.norm(world[pip] - wrist)
            for name, (tip, pip) in FINGER_JOINTS.items()}

# Cuánto está doblado cada dedo: recorrido de sus tres tramos entre la distancia
# directa de la base a la punta. Recto sale cerca de 1 y doblado bastante más
def finger_curl(hand):
    world = hand["world"]
    curl = {}
    for name, (mcp, pip, dip, tip) in FINGER_CHAINS.items():
        path = (np.linalg.norm(world[pip] - world[mcp])
                + np.linalg.norm(world[dip] - world[pip])
                + np.linalg.norm(world[tip] - world[dip]))
        chord = max(np.linalg.norm(world[tip] - world[mcp]), 1e-6)
        curl[name] = path / chord
    return curl

# Cada gesto es una función que mira a un jugador y dice si lo hace en este frame
def is_kiss(player):
    return player["face"]["blendshapes"]["mouthPucker"] > KISS_THRESHOLD

def is_brows_up(player):
    return player["face"]["blendshapes"]["browInnerUp"] > BROWS_UP_THRESHOLD

# Pistola: índice apuntando y el resto de dedos doblados. Se prueban dos formas de
# medirlo, porque ninguna funciona sola en todas las orientaciones de la mano
def is_pistol(player):
    others = ("middle", "ring", "pinky")

    for hand in player["hands"]:
        curl = finger_curl(hand)
        ratio = finger_ratios(hand)

        # De frente: el índice está recto y los otros tres claramente doblados
        front = (curl["index"] < FINGER_STRAIGHT_MAX
                 and all(curl[name] > FINGER_CURLED_MIN for name in others))

        # De lado: el índice se aleja de la muñeca mucho más que el resto
        side = (ratio["index"] > FINGER_EXTENDED_RATIO
                and ratio["index"] - max(ratio[name] for name in others) > POINTING_MARGIN)

        if front or side:
            return True
    return False

# Gestos disponibles
GESTURES = {"kiss": is_kiss, "brows": is_brows_up, "pistol": is_pistol}

# Controla cuántos frames seguidos lleva activo un gesto de un jugador
class GestureState:
    def __init__(self, hold_frames=GESTURE_HOLD_FRAMES):
        self.hold_frames = hold_frames
        self.count = 0
        self.start_time = None

    # Devuelve el instante en que empezó el gesto si ya se mantiene el tiempo
    # exigido, y None en caso contrario
    def update(self, active, now):
        if active:
            if self.count == 0:
                self.start_time = now
            self.count += 1
        else:
            self.count = 0
            self.start_time = None

        if self.count >= self.hold_frames:
            return self.start_time
        return None

# Lógica del juego

In [59]:
GESTURE_NAMES = {"kiss": "BESO", "brows": "CEJAS", "pistol": "PISTOLA"}

# Modos del menú: tecla -> (título, gesto fijo o tipo de aleatorio)
MODES = {
    "1": ("Solo beso", "kiss"),
    "2": ("Solo cejas", "brows"),
    "3": ("Solo pistola", "pistol"),
    "4": ("Aleatorio, avisando antes", "random_before"),
    "5": ("Aleatorio, avisando al disparar", "random_at_draw"),
}

TEXT_WHITE = (255, 255, 255)
TEXT_GOLD = (255, 200, 60)
TEXT_RED = (255, 80, 60)

class Game:
    def __init__(self):
        self.state = "MENU"
        self.mode = None
        self.scores = [0, 0]
        self.gesture = None
        self.gesture_states = [GestureState(), GestureState()]
        self.state_start = 0.0
        self.last_now = 0.0
        self.wait_duration = 0.0
        self.draw_time = 0.0
        self.result_winner = None
        self.result_title = ""
        self.result_detail = ""
        self.result_shooter = None      # quien ha disparado bien (para el "¡BANG!")
        self.result_hat_losers = []     # a quién se le va el sombrero

    def set_state(self, state, now):
        self.state = state
        self.state_start = now

    def start_match(self, mode_key, now):
        # Sin manos no se puede jugar el modo de la pistola
        if MODES[mode_key][1] == "pistol" and not USE_HANDS:
            return
        self.mode = mode_key
        self.scores = [0, 0]
        self.start_round(now)

    def start_round(self, now):
        mode_type = MODES[self.mode][1]
        if mode_type in GESTURES:
            self.gesture = mode_type
        else:
            pool = [name for name in GESTURES if USE_HANDS or name != "pistol"]
            self.gesture = random.choice(pool)

        self.gesture_states = [GestureState(), GestureState()]
        self.wait_duration = random.uniform(*WAIT_RANGE)
        # Solo el modo "avisando antes" tiene una fase previa para enseñar el gesto
        self.set_state("ANNOUNCE" if mode_type == "random_before" else "WAIT", now)

    def finish_round(self, winner, title, detail, now, shooter=None, hat_losers=()):
        if winner is not None:
            self.scores[winner] += 1
        self.result_winner = winner
        self.result_title = title
        self.result_detail = detail
        self.result_shooter = shooter
        self.result_hat_losers = list(hat_losers)
        self.set_state("RESULT", now)

    def resolve_false_start(self, cheaters, now):
        if len(cheaters) == 2:
            self.finish_round(None, "¡Salida en falso de los dos!", "Se repite la ronda",
                              now, hat_losers=[0, 1])
        else:
            loser = cheaters[0]
            winner = 1 - loser
            self.finish_round(winner, f"¡Salida en falso de J{loser + 1}!",
                              f"Punto para J{winner + 1}", now, hat_losers=[loser])

    # Para cada jugador devuelve el instante en que empezó el gesto de la ronda
    # (None si no lo está haciendo o aún no se mantiene el tiempo exigido)
    def read_gestures(self, players, now):
        detect = GESTURES[self.gesture]
        fired = []
        for i in range(2):
            active = i < len(players) and detect(players[i])
            fired.append(self.gesture_states[i].update(active, now))
        return fired

    def update(self, players, now):
        dt = now - self.last_now
        self.last_now = now

        if self.state in ("MENU", "MATCH_END"):
            return

        # Antes del disparo, la ronda se pausa si falta algún jugador
        if self.state in ("ANNOUNCE", "WAIT") and len(players) < 2:
            self.state_start += dt
            self.gesture_states = [GestureState(), GestureState()]
            return

        elapsed = now - self.state_start

        if self.state == "ANNOUNCE":
            if elapsed >= ANNOUNCE_SECONDS:
                self.set_state("WAIT", now)

        elif self.state == "WAIT":
            fired = self.read_gestures(players, now)
            cheaters = [i for i in range(2) if fired[i] is not None]
            if cheaters:
                self.resolve_false_start(cheaters, now)
            elif elapsed >= self.wait_duration:
                self.draw_time = now
                self.set_state("DRAW", now)

        elif self.state == "DRAW":
            fired = self.read_gestures(players, now)
            limit = self.draw_time + MIN_REACTION_SECONDS
            # Un gesto que empezó antes de poder reaccionar es una anticipación
            cheaters = [i for i in range(2) if fired[i] is not None and fired[i] < limit]
            shooters = [i for i in range(2) if fired[i] is not None and fired[i] >= limit]

            if cheaters:
                self.resolve_false_start(cheaters, now)
            elif len(shooters) == 2 and fired[0] == fired[1]:
                self.finish_round(None, "¡Empate!", "Se repite la ronda", now)
            elif shooters:
                winner = min(shooters, key=lambda i: fired[i])
                ms = (fired[winner] - self.draw_time) * 1000
                self.finish_round(winner, f"¡J{winner + 1} gana la ronda!",
                                  f"Tiempo de reacción: {ms:.0f} ms", now,
                                  shooter=winner, hat_losers=[1 - winner])
            elif elapsed >= DRAW_TIMEOUT:
                self.finish_round(None, "Nadie ha disparado", "Se repite la ronda", now)

        elif self.state == "RESULT":
            if elapsed >= RESULT_SECONDS:
                if max(self.scores) >= ROUNDS_TO_WIN:
                    self.set_state("MATCH_END", now)
                else:
                    self.start_round(now)

    def handle_key(self, key, now):
        ch = chr(key).lower()
        if ch == "m":
            self.state = "MENU"
        elif self.state == "MENU" and ch in MODES:
            self.start_match(ch, now)
        elif self.state == "MATCH_END" and ch == "r":
            self.start_match(self.mode, now)

    # Encola los textos de la pantalla actual (se pintan luego con flush_text)
    def draw_hud(self, frame_shape):
        h = frame_shape[0]

        if self.state == "MENU":
            queue_text("QUICK DRAW", 56, TEXT_GOLD, 20)
            queue_text("Elige un modo pulsando su número", 22, TEXT_WHITE, 90)
            for k, (key, (title, mode_type)) in enumerate(MODES.items()):
                suffix = " (manos desactivadas)" if mode_type == "pistol" and not USE_HANDS else ""
                queue_text(f"{key}  {title}{suffix}", 26, TEXT_WHITE, 140 + 40 * k)
            return

        mode_type = MODES[self.mode][1]
        queue_text(f"J1  {self.scores[0]} - {self.scores[1]}  J2", 30, TEXT_WHITE, 10)

        # El gesto se muestra siempre, salvo en el modo que lo oculta hasta el disparo
        hidden = mode_type == "random_at_draw" and self.state in ("WAIT", "DRAW")
        if not hidden and self.state != "MATCH_END":
            queue_text(f"Gesto: {GESTURE_NAMES[self.gesture]}", 22, TEXT_GOLD, 50)

        mid = h // 2 - 60
        if self.state == "ANNOUNCE":
            queue_text("Esta ronda:", 30, TEXT_WHITE, mid - 10)
            queue_text(f"¡{GESTURE_NAMES[self.gesture]}!", 64, TEXT_GOLD, mid + 30)
        elif self.state == "WAIT":
            queue_text("Prepárense...", 48, TEXT_WHITE, mid)
        elif self.state == "DRAW":
            queue_text("¡FUEGO!", 80, TEXT_RED, mid - 10)
            if mode_type == "random_at_draw":
                queue_text(f"¡{GESTURE_NAMES[self.gesture]}!", 56, TEXT_GOLD, mid + 80)
        elif self.state == "RESULT":
            color = TEXT_WHITE if self.result_winner is None else PLAYER_COLORS[self.result_winner]
            queue_text(self.result_title, 36, color, mid)
            queue_text(self.result_detail, 28, TEXT_WHITE, mid + 55)
        elif self.state == "MATCH_END":
            winner = self.scores.index(max(self.scores))
            queue_text(f"¡J{winner + 1} GANA EL DUELO!", 44, PLAYER_COLORS[winner], mid)
            queue_text("R: revancha    M: menú", 26, TEXT_WHITE, mid + 60)

# Efectos y ambiente

In [ ]:
# Estrella de la explosión "¡BANG!": 14 puntas algo irregulares, de radio 1
_rng = np.random.RandomState(3)
_angles = np.linspace(0, 2 * np.pi, 28, endpoint=False)
_radii = np.where(np.arange(28) % 2 == 0, _rng.uniform(0.9, 1.1, 28), 0.55)
BURST_UNIT = np.stack([np.cos(_angles) * _radii, np.sin(_angles) * _radii], axis=1)

# Matriz de color del efecto sepia (para canales BGR)
SEPIA_MATRIX = np.array([[0.131, 0.534, 0.272],
                         [0.168, 0.686, 0.349],
                         [0.189, 0.769, 0.393]], dtype=np.float32)

# Partículas de polvo que el viento arrastra hacia la derecha
class Dust:
    def __init__(self, shape):
        h, w = shape[:2]
        rng = np.random.RandomState(7)
        self.size = (w, h)
        self.pos = np.stack([rng.uniform(0, w, DUST_COUNT), rng.uniform(0, h, DUST_COUNT)], axis=1)
        self.speed = rng.uniform(30, 110, DUST_COUNT)
        self.phase = rng.uniform(0, 2 * np.pi, DUST_COUNT)
        self.radius = rng.choice([1, 1, 2], DUST_COUNT)

    def update(self, dt, t):
        w, h = self.size
        self.pos[:, 0] = (self.pos[:, 0] + self.speed * dt) % w
        # Un pequeño vaivén vertical para que no avancen en línea recta
        self.pos[:, 1] = (self.pos[:, 1] + np.sin(t + self.phase) * 12 * dt) % h

    def draw(self, frame):
        for (x, y), r in zip(self.pos, self.radius):
            cv2.circle(frame, (int(x), int(y)), int(r), (150, 185, 215), -1, cv2.LINE_AA)

# Bola de paja que cruza la pantalla de vez en cuando
class Tumbleweed:
    def __init__(self, shape):
        self.h, self.w = shape[:2]
        self.radius = 22
        # Cuerdas fijas sobre el círculo que dan aspecto de maraña
        self.chords = np.random.RandomState(11).uniform(0, 2 * np.pi, (9, 2))
        self.active = False
        self.timer = random.uniform(*TUMBLEWEED_DELAY)
        self.x = 0.0
        self.vx = 0.0
        self.angle = 0.0

    def spawn(self):
        direction = random.choice([-1, 1])
        self.active = True
        self.x = -30.0 if direction == 1 else self.w + 30.0
        self.vx = direction * random.uniform(120, 200)
        self.angle = 0.0

    def update(self, dt):
        if not self.active:
            self.timer -= dt
            if self.timer <= 0:
                self.spawn()
            return

        self.x += self.vx * dt
        # Gira lo mismo que avanza, como una rueda
        self.angle += self.vx * dt / self.radius
        if self.x < -50 or self.x > self.w + 50:
            self.active = False
            self.timer = random.uniform(*TUMBLEWEED_DELAY)

    def draw(self, frame):
        if not self.active:
            return
        # Da pequeños botes mientras rueda
        y = self.h - 60 - abs(np.sin(self.x * 0.04)) * 16
        cx, cy = int(self.x), int(y)
        color = (60, 110, 150)

        cv2.circle(frame, (cx, cy), self.radius, color, 2, cv2.LINE_AA)
        cv2.circle(frame, (cx, cy), int(self.radius * 0.6), color, 1, cv2.LINE_AA)
        for a, b in self.chords:
            p1 = (cx + int(self.radius * np.cos(a + self.angle)), cy + int(self.radius * np.sin(a + self.angle)))
            p2 = (cx + int(self.radius * np.cos(b + self.angle)), cy + int(self.radius * np.sin(b + self.angle)))
            cv2.line(frame, p1, p2, color, 1, cv2.LINE_AA)

# Sombrero que sale despedido con la gravedad y girando
class FlyingHat:
    def __init__(self, sprite, pos, scale, angle, direction):
        self.sprite = sprite
        self.pos = np.array(pos, dtype=np.float32)
        self.scale = scale
        self.angle = angle
        self.vel = np.array([direction * random.uniform(120, 220), -random.uniform(300, 420)],
                            dtype=np.float32)
        self.spin = direction * random.uniform(200, 420)  # grados por segundo

    def update(self, dt):
        self.vel[1] += HAT_GRAVITY * dt
        self.pos += self.vel * dt
        self.angle += self.spin * dt

# Jugadores que ahora mismo no llevan sombrero porque les ha salido volando
def hatless_players(game):
    if game.state == "RESULT":
        return set(game.result_hat_losers)
    if game.state == "MATCH_END":
        return {game.scores.index(min(game.scores))}
    return set()

class Effects:
    def __init__(self, shape):
        self.h, self.w = shape[:2]
        self.dust = Dust(shape)
        self.tumbleweed = Tumbleweed(shape)
        self.vignette = self.make_vignette(self.h, self.w)
        self.flying_hats = []
        self.bangs = []
        self.flash_start = None
        # Reloj propio de los efectos, avanza con el dt de cada frame
        self.time = 0.0

    # Máscara de viñeteado en uint8: 255 en el centro y más oscura hacia las esquinas
    @staticmethod
    def make_vignette(h, w):
        ys, xs = np.mgrid[0:h, 0:w].astype(np.float32)
        d = np.sqrt(((xs - w / 2) / (w / 2)) ** 2 + ((ys - h / 2) / (h / 2)) ** 2) / np.sqrt(2)
        factor = 1 - VIGNETTE_STRENGTH * np.clip(d, 0, 1) ** 2
        return np.repeat((factor * 255).astype(np.uint8)[:, :, None], 3, axis=2)

    # Tono sepia mezclado con la imagen original y esquinas oscurecidas
    def apply_ambient(self, frame):
        sepia = cv2.transform(frame, SEPIA_MATRIX)
        mixed = cv2.addWeighted(frame, 1 - SEPIA_STRENGTH, sepia, SEPIA_STRENGTH, 0)
        return cv2.multiply(mixed, self.vignette, scale=1 / 255)

    def update(self, dt):
        self.time += dt
        self.dust.update(dt, self.time)
        self.tumbleweed.update(dt)

        for hat in self.flying_hats:
            hat.update(dt)
        self.flying_hats = [hat for hat in self.flying_hats if hat.pos[1] < self.h + 150]
        self.bangs = [b for b in self.bangs if self.time - b["start"] <= BANG_SECONDS]

    # Se llama una vez, justo cuando empieza el resultado de una ronda
    def trigger_result(self, game, players):
        shooter = game.result_shooter
        if shooter is not None and shooter < len(players):
            face = players[shooter]["face"]
            width, _ = get_face_metrics(face)
            # La explosión sale hacia el lado donde está el rival
            direction = 1 if shooter == 0 else -1
            nose = face["points"][LM_NOSE_TIP]
            radius = width * 0.65
            cx = float(np.clip(nose[0] + direction * width * 0.9, radius, self.w - radius))
            cy = float(np.clip(nose[1] - width * 0.3, radius, self.h - radius))
            self.bangs.append({"center": np.array([cx, cy]), "radius": radius, "start": self.time})
            self.flash_start = self.time

        for idx in game.result_hat_losers:
            if idx < len(players):
                pos, scale, roll = get_hat_transform(players[idx]["face"])
                # El sombrero sale despedido hacia fuera de la pantalla
                direction = -1 if idx == 0 else 1
                self.flying_hats.append(FlyingHat(HAT_SPRITES[idx], pos, scale, roll, direction))

    # Capa que va detrás de los jugadores
    def draw_background(self, frame):
        self.dust.draw(frame)
        self.tumbleweed.draw(frame)

    # Capa que va por encima de los jugadores
    def draw_foreground(self, frame):
        if self.flash_start is not None:
            alpha = 0.7 * (1 - (self.time - self.flash_start) / FLASH_SECONDS)
            if alpha > 0:
                # Mezcla con blanco: imagen * (1 - alpha) + 255 * alpha
                frame[:] = cv2.convertScaleAbs(frame, alpha=1 - alpha, beta=255 * alpha)
            else:
                self.flash_start = None

        for hat in self.flying_hats:
            blend_sprite(frame, hat.sprite, HAT_ANCHOR, hat.pos, hat.scale, hat.angle)

        for bang in self.bangs:
            age = self.time - bang["start"]
            # La explosión crece rápido al aparecer
            pop = min(1.0, age / 0.12)
            radius = bang["radius"] * (0.3 + 0.7 * pop)
            pts = (BURST_UNIT * radius + bang["center"]).astype(np.int32)
            cv2.fillPoly(frame, [pts], (40, 200, 255), cv2.LINE_AA)
            cv2.polylines(frame, [pts], True, (20, 60, 200), 3, cv2.LINE_AA)

            if pop > 0.5:
                size = int(np.clip(radius * 0.55, 24, 64))
                top = int(bang["center"][1] - size * 0.6)
                queue_text("¡BANG!", size, (220, 30, 30), top, center_x=int(bang["center"][0]))

# Sonidos

In [61]:
# Eje de tiempo en segundos de un sonido de la duración dada
def time_axis(seconds):
    return np.arange(int(seconds * SOUND_RATE)) / SOUND_RATE

# Nota con armónicos que se apaga de forma exponencial (decay = rapidez de la caída).
# La subida y la bajada de 5 y 20 ms evitan los chasquidos al empezar y al cortar
def make_tone(freq, seconds, decay, harmonics=(1.0,)):
    t = time_axis(seconds)
    signal = sum(amp * np.sin(2 * np.pi * freq * (k + 1) * t) for k, amp in enumerate(harmonics))
    attack = np.minimum(t / 0.005, 1.0)
    release = np.minimum((seconds - t) / 0.02, 1.0)
    return signal * attack * release * np.exp(-t / decay)

# Coloca varios sonidos en una misma pista, cada uno en su instante de inicio (s)
def mix(parts, seconds):
    track = np.zeros(int(seconds * SOUND_RATE))
    for start, sound in parts:
        i = int(start * SOUND_RATE)
        end = min(i + len(sound), len(track))
        track[i:end] += sound[:end - i]
    return track

# Campana del "¡FUEGO!": parciales no armónicos y los agudos se apagan antes
def make_bell():
    ratios = (1.0, 2.0, 2.76, 5.4, 8.93)
    decays = (0.45, 0.30, 0.20, 0.10, 0.05)
    amps = (1.0, 0.6, 0.5, 0.3, 0.2)
    return sum(a * make_tone(660 * r, 1.0, d) for r, d, a in zip(ratios, decays, amps))

# Disparo: chasquido de ruido, estampido grave y un golpe que baja de frecuencia
def make_shot():
    t = time_axis(0.7)
    noise = np.random.RandomState(5).uniform(-1, 1, len(t))
    crack = noise * np.exp(-t / 0.03)
    smooth = np.convolve(noise, np.ones(30) / 30, mode="same")
    boom = smooth * 6 * np.exp(-t / 0.15)
    freq = 40 + 120 * np.exp(-t / 0.06)
    thump = np.sin(2 * np.pi * np.cumsum(freq) / SOUND_RATE) * np.exp(-t / 0.12)
    return crack + boom + thump

# Martilleo del revólver: dos clics secos con un toque metálico
def make_cock():
    n = int(0.03 * SOUND_RATE)
    noise = np.random.RandomState(2).uniform(-1, 1, n)
    click = noise * np.exp(-time_axis(0.03) / 0.004) + 0.6 * make_tone(1800, 0.03, 0.008)
    return mix([(0.0, click), (0.14, click * 0.8)], 0.25)

# Zumbador de salida en falso: dos tonos cuadrados que bajan
def make_buzzer():
    def square(freq, seconds):
        t = time_axis(seconds)
        fade = np.minimum(t / 0.005, 1.0) * np.minimum((seconds - t) / 0.03, 1.0)
        return np.sign(np.sin(2 * np.pi * freq * t)) * fade
    return mix([(0.0, square(220, 0.22)), (0.25, square(165, 0.35))], 0.6)

# Ronda sin ganador (empate o nadie dispara): dos notas suaves que bajan
def make_miss():
    return mix([(0.0, make_tone(330, 0.4, 0.15)), (0.22, make_tone(247, 0.6, 0.25))], 0.9)

# Aviso del gesto en el modo 4: dos notas agudas
def make_chime():
    return mix([(0.0, make_tone(784, 0.5, 0.15, (1.0, 0.3))),
                (0.18, make_tone(1047, 0.7, 0.2, (1.0, 0.3)))], 0.9)

# Fanfarria del vencedor: arpegio ascendente con armónicos de trompeta
def make_fanfare():
    brass = (1.0, 0.6, 0.4, 0.25)
    notes = [(0.00, 523.25, 0.18), (0.16, 659.25, 0.18), (0.32, 783.99, 0.18), (0.48, 1046.5, 0.9)]
    return mix([(start, make_tone(freq, dur, dur, brass)) for start, freq, dur in notes], 1.5)

# Cuerda pulsada (algoritmo Karplus-Strong): un trozo de ruido que se va suavizando
# al repetirse, lo que imita una cuerda de guitarra. decay (<1) la apaga antes
def make_pluck(freq, seconds, decay=0.997):
    n = int(SOUND_RATE / freq)
    total = int(seconds * SOUND_RATE)
    y = np.zeros(total + n + 1)
    y[1:n + 1] = np.random.RandomState(int(freq)).uniform(-1, 1, n)
    # Cada bloque de n muestras sale de promediar las vecinas del bloque anterior
    for start in range(n + 1, len(y), n):
        end = min(start + n, len(y))
        y[start:end] = decay * 0.5 * (y[start - n:end - n] + y[start - n - 1:end - n - 1])
    return y[1:total + 1] * np.minimum((seconds - time_axis(seconds)) / 0.02, 1.0)

# Silbido del pistolero: seno con vibrato lento y un poco de segundo armónico
def make_whistle(freq, seconds):
    t = time_axis(seconds)
    vibrato = 1 + 0.012 * np.sin(2 * np.pi * 5.5 * t) * np.minimum(t / 0.3, 1.0)
    phase = 2 * np.pi * np.cumsum(freq * vibrato) / SOUND_RATE
    envelope = np.minimum(t / 0.06, 1.0) * np.minimum((seconds - t) / 0.12, 1.0)
    return (np.sin(phase) + 0.12 * np.sin(2 * phase)) * envelope

# Música de menú: 8 compases en mi menor que se repiten en bucle.
# Bajo al galope (tum, tu-tum), trémolo de guitarra al principio y melodía silbada
def make_music():
    beat = 60 / MUSIC_BPM
    parts = []

    # Nota grave de cada compás: Em, Em, C, D, Em, Em, C, B
    roots = [164.81, 164.81, 130.81, 146.83, 164.81, 164.81, 130.81, 123.47]
    bass = {root: make_pluck(root, 0.35, 0.99) for root in set(roots)}
    for bar, root in enumerate(roots):
        for b in range(4):
            start = (bar * 4 + b) * beat
            for offset, amp in ((0.0, 1.0), (0.5, 0.6), (0.75, 0.8)):
                parts.append((start + offset * beat, amp * bass[root]))

    # Trémolo: la misma nota repetida muy rápido durante los compases 1 y 2
    tremolo = make_pluck(329.63, 0.3)
    for k in range(32):
        parts.append((k * beat / 4, 0.5 * tremolo))

    # Melodía: (compás, tiempo dentro del compás, duración en tiempos, frecuencia)
    melody = [(2, 0, 2, 392.00), (2, 2, 2, 329.63),
              (3, 0, 2, 369.99), (3, 2, 2, 440.00),
              (4, 0, 3, 493.88), (4, 3, 1, 440.00),
              (5, 0, 2, 392.00), (5, 2, 2, 329.63),
              (6, 0, 1, 329.63), (6, 1, 1, 392.00), (6, 2, 2, 523.25),
              (7, 0, 2, 369.99), (7, 2, 2, 311.13)]
    for bar, pos, length, freq in melody:
        start = (bar * 4 + pos) * beat
        parts.append((start, 0.45 * make_whistle(freq, length * beat - 0.05)))

    track = mix(parts, 32 * beat)
    # Fundido final muy corto para que el bucle no dé un chasquido al empalmar
    fade = int(0.06 * SOUND_RATE)
    track[-fade:] *= np.linspace(1, 0, fade)
    return track

SOUND_DIR = os.path.join(tempfile.gettempdir(), "quickdraw_sounds")
SOUND_MAKERS = {"chime": make_chime, "cock": make_cock, "bell": make_bell, "shot": make_shot,
                "buzzer": make_buzzer, "miss": make_miss, "fanfare": make_fanfare}
sound_files = {}

# Guarda el sonido como WAV mono de 16 bits, normalizado para que todos suenen parecido
def save_wav(path, samples, volume=SOUND_VOLUME):
    data = samples / np.max(np.abs(samples)) * volume
    with wave.open(path, "wb") as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(SOUND_RATE)
        f.writeframes((data * 32767).astype(np.int16).tobytes())

# Genera todos los WAV una sola vez (se sobrescriben en cada ejecución)
def init_sounds():
    if not SOUND_ENABLED or winsound is None:
        return
    os.makedirs(SOUND_DIR, exist_ok=True)
    for name, maker in SOUND_MAKERS.items():
        path = os.path.join(SOUND_DIR, f"{name}.wav")
        save_wav(path, maker())
        sound_files[name] = path
    music_path = os.path.join(SOUND_DIR, "music.wav")
    save_wav(music_path, make_music(), MUSIC_VOLUME)
    sound_files["music"] = music_path

# Reproducción en segundo plano. winsound solo admite un sonido a la vez
def play_sound(name):
    if name in sound_files:
        winsound.PlaySound(sound_files[name], winsound.SND_FILENAME | winsound.SND_ASYNC)

# La música se repite en bucle hasta que otro sonido la sustituya
def play_music():
    if "music" in sound_files:
        winsound.PlaySound(sound_files["music"],
                           winsound.SND_FILENAME | winsound.SND_ASYNC | winsound.SND_LOOP)

# Instante en que debe empezar la música tras la fanfarria (None si no hay ninguna pendiente)
_music_due = None

# Se llama en cada frame: arranca la música cuando termina la fanfarria
def update_music():
    global _music_due
    if _music_due is not None and time.perf_counter() >= _music_due:
        _music_due = None
        play_music()

def stop_sounds():
    if winsound is not None:
        winsound.PlaySound(None, winsound.SND_PURGE)

# Elige el sonido según el estado al que acaba de pasar el juego
def play_state_sound(game):
    global _music_due
    # Cualquier cambio de estado cancela una música que estuviera pendiente de arrancar
    _music_due = None

    if game.state == "MENU":
        play_music()
    elif game.state == "ANNOUNCE":
        play_sound("chime")
    elif game.state == "WAIT":
        play_sound("cock")
    elif game.state == "DRAW":
        play_sound("bell")
    elif game.state == "RESULT":
        if game.result_shooter is not None:
            play_sound("shot")
        elif game.result_hat_losers:
            play_sound("buzzer")
        else:
            play_sound("miss")
    elif game.state == "MATCH_END":
        play_sound("fanfare")
        _music_due = time.perf_counter() + FANFARE_SECONDS

init_sounds()

# Bucle principal

In [62]:
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
if not cap.isOpened():
    raise RuntimeError("Could not open the camera")

game = Game()
effects = None   # se crea con el tamaño real del primer frame
start = time.perf_counter()
last_now = start

# Los puntos se pueden mostrar u ocultar con la tecla L
show_landmarks = True

# Estado con el que se comparó por última vez, para detectar cuándo cambia
# y reproducir entonces el sonido que le corresponde
sound_state = game.state

# Arranca la música del menú, porque el juego empieza en MENU
# y ese estado no cambia hasta que se elige un modo
play_state_sound(game)

with FaceLandmarker.create_from_options(face_options) as face_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.flip(frame, 1)
        now = time.perf_counter()
        dt = now - last_now
        last_now = now
        timestamp_ms = int((now - start) * 1000)

        if effects is None:
            effects = Effects(frame.shape)

        # La detección se hace sobre la imagen limpia, antes de aplicar el filtro
        faces = detect_players(face_landmarker, frame, timestamp_ms)
        hands = detect_hands(hand_landmarker, frame, timestamp_ms) if USE_HANDS else []
        players = build_players(faces, hands)

        # Si la ronda acaba de terminar, se lanzan los efectos del resultado
        previous_state = game.state
        game.update(players, now)
        if game.state == "RESULT" and previous_state != "RESULT":
            effects.trigger_result(game, players)

        frame = effects.apply_ambient(frame)
        effects.update(dt)
        effects.draw_background(frame)

        hatless = hatless_players(game)
        for idx, player in enumerate(players):
            draw_accessories(frame, player["face"], HAT_SPRITES[idx], with_hat=idx not in hatless)
            # Se dibujan después de los accesorios para que no queden tapados
            if show_landmarks:
                draw_landmarks(frame, player["face"], PLAYER_COLORS[idx])
                for hand in player["hands"]:
                    draw_hand_landmarks(frame, hand, PLAYER_COLORS[idx])

        effects.draw_foreground(frame)

        game.draw_hud(frame.shape)
        if len(players) < 2 and game.state != "MENU":
            queue_text("Se necesitan dos jugadores", 26, TEXT_WHITE, frame.shape[0] - 40)

        frame = flush_text(frame)
        cv2.imshow("Quick Draw", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # Esc
            break
        elif key == ord('l'):
            show_landmarks = not show_landmarks
        else:
            game.handle_key(key, now)

        # Sonido al cambiar de estado, también cuando el cambio lo provoca una tecla
        if game.state != sound_state:
            play_state_sound(game)
            sound_state = game.state

        # Arranca la música del fin del duelo cuando termina la fanfarria
        update_music()

cap.release()
stop_sounds()
cv2.destroyAllWindows()